In [ ]:
# shots_train_test.ipynb  (MATCHED TO bsnn_simulation FOR IDEAL CASE)


import os
import platform
import torch
torch.set_num_threads(16)
import numpy as np
from sklearn import svm
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from common_functions import *
from data_loader import *
import json
from fractions import Fraction
import random
from sklearn.model_selection import train_test_split
import pickle

device = torch.device("cpu")

MNIST_LIKE_DATASETS = (
    'cifar10', 'MNIST', 'MNISTresized', 'fashionMNIST', 'MNIST01', 'MNIST17',
    'fashionMNIST_Tshirt_dress', 'organMNISTa', 'organMNISTb'
)


QuantumKernelNN_bsnn = QuantumKernelNN
photonic_Gram_calculator_bsnn = photonic_Gram_calculator
convert_data_bsnn = convert_data
SVM_acc_test_bsnn = SVM_acc_test
SVM_acc_train_bsnn = SVM_acc_train


# ============================================================
# Shot-noise helper
# ============================================================
def apply_shot_noise_to_gram(K_true: torch.Tensor, M: int, eps: float = 1e-12) -> torch.Tensor:
    """
    Sample each off-diagonal kernel entry from Binomial(M, p)/M.
    Diagonal is fixed to 1.
    """
    if M <= 0:
        raise ValueError("M (shots per kernel entry) must be positive.")

    device_local = K_true.device
    Nloc = K_true.shape[0]
    K_true = torch.clamp(K_true, 0.0, 1.0)

    triu_i, triu_j = torch.triu_indices(Nloc, Nloc, device=device_local)
    mask = triu_i != triu_j
    i_up = triu_i[mask]
    j_up = triu_j[mask]

    p = K_true[i_up, j_up]

    binom = torch.distributions.Binomial(total_count=float(M), probs=p)
    counts = binom.sample()
    K_hat_vals = counts / float(M)

    K_hat = torch.zeros_like(K_true)
    K_hat[i_up, j_up] = K_hat_vals
    K_hat[j_up, i_up] = K_hat_vals
    K_hat.fill_diagonal_(1.0)

    d = torch.clamp(torch.diag(K_hat), min=eps)
    K_hat = K_hat / torch.sqrt(d[:, None] * d[None, :])
    K_hat.fill_diagonal_(1.0)

    return K_hat


def noisy_gram_for_loss_ste(K_true: torch.Tensor, M: int) -> torch.Tensor:
    """
    Straight-through estimator:
    - forward uses shot-noisy Gram
    - backward uses gradients of K_true
    """
    K_hat = apply_shot_noise_to_gram(K_true, M)
    return K_true + (K_hat - K_true).detach()

# ============================================================
# Save-path helpers
# ============================================================
def shot_folder_path(root_folder, Mshots):
    return os.path.join(root_folder, f"shots_{int(Mshots)}")


def epoch_results_path(shot_folder, N, modes, depth, num_photons, state_name, epoch):
    return os.path.join(
        shot_folder,
        f"accuracies_N{N}_modes{modes}_depth{depth}_photons{num_photons}_state_{state_name}_epochs{epoch}.json"
    )


def read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def write_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=4)


def ensure_results_schema(results, meta):
    if results is None:
        results = {}

    if "meta" not in results:
        results["meta"] = {}
    for k, v in meta.items():
        if k not in results["meta"]:
            results["meta"][k] = v

    if "quantum" not in results:
        results["quantum"] = {}

    q = results["quantum"]
    q.setdefault("q_true_acc_test", [])
    q.setdefault("q_true_acc_train", [])
    q.setdefault("q_prime_acc_test", [])
    q.setdefault("q_prime_acc_train", [])
    q.setdefault("q_loss", [])

    return results

def set_value_at_index(lst, idx, value):
    while len(lst) < idx:
        lst.append(None)
    if len(lst) == idx:
        lst.append(value)
    else:
        lst[idx] = value


def get_completed_iterations_for_shot(root_folder, Mshots, N, modes, depth, num_photons, state_name, final_epoch):
    sfolder = shot_folder_path(root_folder, Mshots)
    rpath = epoch_results_path(sfolder, N, modes, depth, num_photons, state_name, final_epoch)

    if os.path.exists(rpath):
        data = read_json(rpath)
        return len(data.get("quantum", {}).get("q_prime_acc_test", []))
    return 0


# ============================================================
# Main experiment
# ============================================================
def main_sampling(
    N, test_portion, num_epochs, n_iterations, data_type,
    root_folder, split_cache_root, modes, depth,
    X_data, y_data, X_data_train, X_data_test, y_data_train, y_data_test,
    num_photons,
    shots_list,
    init_state=None, state_name=None
):
    print("DEBUG main_sampling() got N =", N)

    device_local = torch.device("cpu")

    if num_photons > modes:
        return

    state_name, init_state = get_leftmost_state(modes, num_photons)

    # ============================================================
    # IMPORTANT:
    # We now train SEPARATELY for each shot budget M.
    # ============================================================
    for Mshots in shots_list:
        Mshots = int(Mshots)

        print("\n" + "-" * 80)
        print(f"Training separate model for M = {Mshots} shots")
        print("-" * 80)

        start_jj = get_completed_iterations_for_shot(
            root_folder=root_folder,
            Mshots=Mshots,
            N=N,
            modes=modes,
            depth=depth,
            num_photons=num_photons,
            state_name=state_name,
            final_epoch=num_epochs - 1
        )

        if start_jj > 0:
            print(f"Resuming shots={Mshots} from iteration jj={start_jj}")

        for jj in range(start_jj, n_iterations):
            seed_model = SEED + jj

            set_all_seeds(seed_model)
            rng_state = get_rng_state()

            # -----------------------
            # Same split logic as bsnn
            # -----------------------
            if data_type in MNIST_LIKE_DATASETS:
                idx_tr, idx_te = get_or_make_mnist_sep_indices(
                    folder_path=split_cache_root,
                    data_type=data_type,
                    N=N,
                    test_portion=test_portion,
                    jj=jj,
                    n_train_total=len(X_data_train),
                    n_test_total=len(X_data_test),
                    seed_base=SEED
                )

                (X_raw, X_raw_train, X_raw_test, y_labels, y_train, y_test,
                 subset_indexes_train, subset_indexes_test, global_indexes_train, global_indexes_test) = \
                    build_from_mnist_separate(
                        X_data_train, y_data_train, X_data_test, y_data_test,
                        idx_tr, idx_te
                    )
            else:
                total_len = len(X_data)
                idx_subset, idx_train_global, idx_test_global = get_or_make_split_indices(
                    folder_path=split_cache_root,
                    data_type=data_type,
                    N=N,
                    test_portion=test_portion,
                    jj=jj,
                    total_len=total_len,
                    seed_base=SEED
                )

                (X_raw, X_raw_train, X_raw_test, y_labels, y_train, y_test,
                 subset_indexes_train, subset_indexes_test, global_indexes_train, global_indexes_test) = \
                    build_from_global_split(
                        X_data, y_data, idx_subset, idx_train_global, idx_test_global
                    )

            print(f"modes={modes}, depth={depth}, photons={num_photons}, shots={Mshots}, iter={jj}")

            num_train = int((1 - test_portion) * N)
            z_ij = zij_calc(num_train, y_train)

            num_classes = len(np.unique(y_labels))
            _, num_circ_params = pad_input(X_raw, modes, depth, init_state)

            if num_classes > 2:
                classifier = OneVsRestClassifier(SVC(kernel='precomputed'))
            else:
                classifier = svm.SVC(kernel='precomputed', verbose=False)

            nn_in_dim = len(X_raw[0])
            nn_out_dim = num_circ_params

            set_rng_state(rng_state)
            print(torch.rand(3)[:3], np.random.rand(3))

            # fresh model for this shot budget
            model = QuantumKernelNN_bsnn(
                input_dim=nn_in_dim,
                output_dim=nn_out_dim,
                init_state=init_state,
                modes=modes,
                depth=depth,
                block_a=128,
                block_b=128,
                chunk_subsets=1 << 13
            ).to(device_local)

            optimizer = optim.Adam(model.parameters(), lr=0.001)
            criterion = nn.MSELoss()

            X_tensor = torch.stack([
                x.clone().detach().to(device_local).requires_grad_(True)
                for x in X_raw_train
            ])

            sfolder = shot_folder_path(root_folder, Mshots)
            os.makedirs(sfolder, exist_ok=True)

            for epoch in range(num_epochs):
                optimizer.zero_grad()

                # --------------------------------------------------
                # Ideal train Gram from model
                # --------------------------------------------------
                X_prime_train, Gram_prime_train_true = model(X_tensor)

                # --------------------------------------------------
                # Shot-noisy train Gram used IN THE LOSS
                # --------------------------------------------------
                Gram_prime_train_noisy = noisy_gram_for_loss_ste(
                    Gram_prime_train_true, Mshots
                )

                loss = criterion(Gram_prime_train_noisy, z_ij)
                loss.backward()
                optimizer.step()

                X_prime_train = X_prime_train.detach()
                Gram_prime_train_true = Gram_prime_train_true.detach()

                # test embedding path
                X_prime, X_prime_test = convert_data_bsnn(
                    N, X_raw_test, subset_indexes_test, subset_indexes_train, X_prime_train, model
                )

                print(f"[shots {Mshots}][iter {jj}] Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}")

                # ideal Grams for reporting
                K_test_true, _ = photonic_Gram_calculator_bsnn(
                    X_prime, modes, depth, init_state
                )
                K_train_true, _ = photonic_Gram_calculator_bsnn(
                    X_prime_train, modes, depth, init_state
                )

                # noisy Grams for reporting
                K_test_hat = apply_shot_noise_to_gram(K_test_true, Mshots)
                K_train_hat = apply_shot_noise_to_gram(K_train_true, Mshots)

                true_acc_test = float(
                    SVM_acc_test_bsnn(
                        K_test_true, classifier,
                        subset_indexes_train, subset_indexes_test,
                        y_train, y_test
                    )[0]
                )

                true_acc_train = float(
                    SVM_acc_train_bsnn(
                        X_prime_train, y_train, K_train_true, classifier
                    )[0]
                )

                acc_test = float(
                    SVM_acc_test_bsnn(
                        K_test_hat, classifier,
                        subset_indexes_train, subset_indexes_test,
                        y_train, y_test
                    )[0]
                )

                acc_train = float(
                    SVM_acc_train_bsnn(
                        X_prime_train, y_train, K_train_hat, classifier
                    )[0]
                )

                results_file = epoch_results_path(
                    sfolder, N, modes, depth, num_photons, state_name, epoch
                )

                meta = {
                    "data_type": str(data_type),
                    "N": int(N),
                    "test_portion": float(test_portion),
                    "modes": int(modes),
                    "depth": int(depth),
                    "num_photons": int(num_photons),
                    "state_name": str(state_name),
                    "shots_per_kernel_entry": int(Mshots),
                    "all_shots_list": [int(s) for s in shots_list],
                    "n_iterations_target": int(n_iterations),
                    "seed_base": int(SEED),
                    "seed_effective": int(SEED + jj),
                    "split_cache_root": str(split_cache_root),
                    "training_is_shot_noisy": True,
                    "loss_uses_ste": True,
                }

                if os.path.exists(results_file):
                    final_results = read_json(results_file)
                else:
                    final_results = None

                final_results = ensure_results_schema(final_results, meta)

                set_value_at_index(final_results["quantum"]["q_loss"], jj, float(loss.item()))
                set_value_at_index(final_results["quantum"]["q_true_acc_test"], jj, true_acc_test)
                set_value_at_index(final_results["quantum"]["q_true_acc_train"], jj, true_acc_train)
                set_value_at_index(final_results["quantum"]["q_prime_acc_test"], jj, acc_test)
                set_value_at_index(final_results["quantum"]["q_prime_acc_train"], jj, acc_train)

                write_json(results_file, final_results)

# ============================================================
# Run
# ============================================================
if __name__ == "__main__":

    for data_type in ['MNIST']:
        X_data, y_data, X_train, X_test, y_train, y_test = prepare_data(data_type)

        if data_type == 'MNIST':
            N = 5000
            SEED = 135
            test_portion = 1 / 7
            max_mode = 12
        else:
            SEED = 110
            N = len(X_data)
            test_portion = 0.2
            max_mode = 6

        portion = Fraction(test_portion).limit_denominator()
        denominator = portion.denominator
        numerator = portion.numerator

        if platform.system() == 'Windows':
            base_path = r'E:\BSNN'
        else:
            home = os.path.expanduser("~")
            base_path = os.path.join(home, "BSNN")

        num_epochs = 50
        n_iterations = 5

        
        photon_range = range(3, 4, 1)
        modes_range = [max_mode]
        depth_range = modes_range

        shots_list = [10,100,1000]

        # Save everything here
        root_folder = os.path.join(
            base_path,
            f"{data_type}_test_portion_{numerator}_of_{denominator}_shots"
        )
        os.makedirs(root_folder, exist_ok=True)

        # Keep split cache here too
        split_cache_root = root_folder
        os.makedirs(os.path.join(split_cache_root, "splits"), exist_ok=True)

        for Mshots in shots_list:
            os.makedirs(shot_folder_path(root_folder, Mshots), exist_ok=True)

        for num_photons in photon_range:
            for modes in modes_range:
                for depth in depth_range:
                    print("\n" + "=" * 80)
                    print(f"Sampling sweep: N={N} modes={modes} depth={depth} photons={num_photons}")
                    print(f"shots_list = {shots_list}")
                    print(f"root_folder = {root_folder}")
                    print(f"split_cache_root = {split_cache_root}")
                    print("=" * 80 + "\n")

                    main_sampling(
                        N, test_portion, num_epochs, n_iterations, data_type,
                        root_folder, split_cache_root, modes, depth,
                        X_data, y_data, X_train, X_test, y_train, y_test,
                        num_photons=num_photons,
                        shots_list=shots_list
                    )

    print("Done!!!")